# Продвинутое домашнее задание: Доменная адаптация энкодерной модели для семантического поиска

В рамках этого домашнего задания вы пройдете полный цикл адаптации предобученной англоязычной энкодерной модели `sentence-transformers/all-MiniLM-L6-v2` (обратите внимание: модель обучалась на английских данных, что делает задачу адаптации к русскому языку особенно показательной) для работы с русскоязычными текстами. Ваша цель - улучшить качество семантического поиска и визуально оценить изменения в пространстве эмбеддингов.

**Важное предупреждение:** Доменная адаптация - это сложный процесс, требующий тщательного подбора гиперпараметров и качественных данных. В рамках выполнения данного ДЗ в образовательных целях, скорее всего, вы получите лишь незначительное улучшение метрик поиска. Это абсолютно нормально. Главная цель - освоить методологию и пайплайн дообучения.

In [ ]:
!pip install sentence-transformers datasets faiss-cpu umap-learn matplotlib seaborn pandas numpy

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import faiss
import umap
from tqdm.auto import tqdm
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## Часть 1. Визуализация пространства эмбеддингов до адаптации (20 баллов)

Перед тем как что либо улучшать, необходимо понять, с чем мы имеем дело. Мы визуализируем эмбеддинги коротких текстов, чтобы увидеть, насколько хорошо базовая модель разделяет их по смыслу.

**Задание:**
1. Загрузите датасет `ai-forever/headline-classification` (сплит `test`, возьмите первые 1000 примеров: `split="test[:1000]"`). Датасет содержит колонки `text` (заголовок новости на русском языке, строка), `label` (номер класса, число) и `label_text` (название тематики, строка: спорт, политика, экономика и т.д.). Для получения эмбеддингов используйте колонку `text`, для раскраски точек на графике - колонку `label_text`.
2. Инициализируйте базовую модель `sentence-transformers/all-MiniLM-L6-v2`.
3. Получите эмбеддинги для текстов из датасета.
4. Напишите функцию для понижения размерности эмбеддингов до 2D с использованием алгоритма UMAP.
5. Постройте scatter plot, раскрасив точки в соответствии с метками классов.

**Подсказки:**
* Обязательно зафиксируйте `random_state` в UMAP для воспроизводимости.

### Важно
Вы можете выбрать другие технологии или набор данных для решения задачи, аргументировав свое решение

In [ ]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
base_model = SentenceTransformer(model_name)

# Загрузка датасета для визуализации
vis_dataset = load_dataset("ai-forever/headline-classification", split="test[:1000]")
vis_texts = vis_dataset["text"]
vis_labels = vis_dataset["label_text"]

# Ваш код здесь: получение эмбеддингов базовой моделью
base_embeddings = None # TODO

def plot_embeddings(embeddings, labels, title):
    # Ваш код здесь: UMAP и scatter plot
    pass

# plot_embeddings(base_embeddings, vis_labels, "UMAP: Базовая модель")

## Часть 2. Оценка базового качества поиска (25 баллов)

Теперь замерим метрики семантического поиска до доменной адаптации.

**Задание:**
1. Загрузите датасет `ai-forever/rubq-retrieval`. Этот датасет состоит из трех отдельных подмножеств, которые нужно загрузить отдельно:
   * Вопросы: подмножество (config) `queries`, сплит `queries`. Содержит колонки `_id` (строка) и `text` (строка).
   * База знаний: подмножество (config) `corpus`, сплит `corpus`. Содержит колонки `_id` (строка), `title` (строка) и `text` (строка).
   * Разметка релевантности (qrels): подмножество (config) `default`, сплит `test`. Содержит колонки `query-id` (строка), `corpus-id` (строка) и `score` (число).
2. Напишите функцию `evaluate_retrieval`, которая принимает на вход модель, вопросы (`queries`), базу знаний (`corpus`) и разметку релевантности (`qrels`), и возвращает метрики Recall@5 и Recall@10.
3. Для ускорения поиска реализуйте индекс на основе `faiss.IndexFlatIP`.
4. Замерьте и зафиксируйте базовое качество поиска.

**Подсказки и рекомендации по оценке:**
* Поскольку мы используем `faiss.IndexFlatIP` (поиск по скалярному произведению), векторы перед добавлением в индекс необходимо L2-нормализовать. Укажите `normalize_embeddings=True` при вызове метода `encode`.
* **Как считать Recall@K:**
  1. Для каждого вопроса (колонка `text` из `queries`) получите топ-K ближайших документов из базы знаний (колонка `text` из `corpus`) с помощью `faiss.Index.search`.
  2. Для этого же вопроса найдите все правильные (релевантные) документы в таблице `qrels` (строки, где колонка `query-id` совпадает с `_id` вопроса, а колонка `corpus-id` содержит ID правильных документов). Обратите внимание: у многих вопросов правильных документов несколько.
  3. Вычислите Recall@K для текущего вопроса по формуле: (количество правильных документов, попавших в ваш топ-K) / (общее количество правильных документов для этого вопроса из `qrels`).
  4. Итоговый Recall@K - это среднее арифметическое значений Recall@K по всем вопросам.
* **Маппинг идентификаторов:** Метод `faiss.Index.search` возвращает целочисленные индексы строк (от 0 до N-1). Однако в таблице `qrels` в колонке `corpus-id` лежат строковые ID документов. Важно: эти строковые ID из колонки `_id` корпуса не совпадают с порядковыми номерами строк (в нумерации есть пропуски). Поэтому простое преобразование `str(faiss_index)` даст неверные результаты. Обязательно постройте словарь-маппинг: `id_map = {i: doc_id for i, doc_id in enumerate(corpus['_id'])}` и используйте его для перевода индексов FAISS в строковые `corpus-id`.

### Важно
Вы можете выбрать другие технологии или набор данных для решения задачи, аргументировав свое решение

In [ ]:
# Загрузка датасета для оценки
eval_queries = load_dataset("ai-forever/rubq-retrieval", "queries", split="queries")
eval_corpus = load_dataset("ai-forever/rubq-retrieval", "corpus", split="corpus")
eval_qrels = load_dataset("ai-forever/rubq-retrieval", "default", split="test")

def evaluate_retrieval(model, queries_ds, corpus_ds, qrels_ds, top_k=[5, 10]):
    # Подсказка (пошаговый план):
    # 1. Закодировать corpus_ds["text"] (normalize_embeddings=True)
    # 2. Создать faiss.IndexFlatIP(dimension) и добавить векторы корпуса
    # 3. Закодировать queries_ds["text"] (normalize_embeddings=True)
    # 4. Сделать index.search(query_embeddings, max(top_k))
    # 5. Перевести индексы FAISS в строковые corpus-id через id_map (см. подсказку выше)
    # 6. Рассчитать Recall@K: среднее по вопросам от (найдено релевантных в топ-K / всего релевантных)
    
    # Ваш код здесь:
    pass

# base_metrics = evaluate_retrieval(base_model, eval_queries, eval_corpus, eval_qrels)
# print("Базовые метрики:", base_metrics)

## Часть 3. Доменная адаптация через TripletLoss (30 баллов)

Самая важная часть задания. Мы будем дообучать модель на русскоязычных данных с использованием функции потерь TripletLoss.

**Задание:**
1. Загрузите датасет `deepvk/ru-HNP` (сплит `train`, для ускорения можете взять первые 20000 примеров). Этот датасет содержит колонки `query` (якорь, строка), `pos` (список позитивных парафраз) и `neg` (список сложных негативных примеров).
2. Подготовьте данные для обучения. Сформируйте обучающую выборку в формате `InputExample(texts=[anchor, positive, negative])` для библиотеки `sentence-transformers`. В качестве `anchor` берите строку из колонки `query`, в качестве `positive` - первый элемент из списка в колонке `pos`, в качестве `negative` - первый элемент из списка в колонке `neg`. 
3. Настройте `DataLoader` (batch_size=16 или 32) и выберите функцию потерь `TripletLoss` (доступна как `losses.TripletLoss`).
4. Запустите дообучение модели на 3 эпохи.
5. Опубликуйте модель.

**Подсказки:**
* Если вы столкнулись с нехваткой памяти, уменьшите `batch_size`.

### Важно
Вы можете выбрать другие технологии или набор данных для решения задачи, аргументировав свое решение


In [ ]:
# Загрузка обучающего датасета
train_dataset = load_dataset("deepvk/ru-HNP", split="train[:20000]")

train_examples = []
# Ваш код здесь: формирование train_examples из InputExample

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)

# Инициализируем модель заново для чистоты эксперимента (или используем base_model)
finetuned_model = SentenceTransformer(model_name)

# Ваш код здесь: настройка TripletLoss и запуск model.fit()
train_loss = None # TODO: losses.TripletLoss

# finetuned_model.fit(train_objectives=[(train_dataloader, train_loss)], epochs=1, warmup_steps=100)

## Часть 4. Оценка качества после адаптации (25 баллов)

Проверим, дало ли наше дообучение результат.

**Задание:**
1. Снова вызовите функцию `evaluate_retrieval` на датасете `ai-forever/rubq-retrieval`, но теперь передайте в неё дообученную модель.
2. Сравните метрики Recall@K до и после дообучения. Сделайте письменный вывод.
3. Повторите процесс из Части 1: визуализируйте эмбеддинги датасета `ai-forever/headline-classification` с помощью дообученной модели. Сравните два графика. Стали ли кластеры более разделимыми?

### Важно
Вы можете выбрать другие технологии или набор данных для решения задачи, аргументировав свое решение

In [ ]:
# Ваш код здесь: оценка Recall@K для дообученной модели
# finetuned_metrics = evaluate_retrieval(finetuned_model, eval_queries, eval_corpus, eval_qrels)
# print("Метрики после дообучения:", finetuned_metrics)

In [ ]:
# Ваш код здесь: получение эмбеддингов дообученной моделью и визуализация
finetuned_embeddings = None # TODO

# plot_embeddings(finetuned_embeddings, vis_labels, "UMAP: Дообученная модель")

### Выводы

Напишите краткий вывод по результатам эксперимента. Улучшились ли метрики? Как изменилась визуализация кластеров? Почему доменная адаптация на парафразах могла дать именно такой результат на задаче QA-поиска?

**Ваш ответ:**

(напишите текст здесь)